In [0]:
%sql
-- !!! ---- ESTE SCRIPT NAO SERA MAIS UTILIZADO ---- !!!
-- SCRIPT: 6.1_analise_absenteismo_parlamentar
-- LOCAL: 2_silver/src/6_monitor_presenca/
-- OBJETIVO: Monitora as ausencias. Usa as tabelas gold_atlas_frentes_parlamentares e bronze_presenca_votacoes
-- ENTREGA: Desenvolver uma visão sobre junção entre eventos e votações para medir ausências em votações
-- EXCLUIR: Substituído pelo 6.1_analise_absenteismo na camada Gold
-- !!! ---- ESTE SCRIPT NAO SERA MAIS UTILIZADO ---- !!!

 -- Tabela Gold de Absenteísmo
CREATE OR REPLACE TABLE gold_monitor_absenteismo AS
WITH total_deputados AS (
  -- Lista única de deputados que temos no Atlas
  SELECT DISTINCT id_deputado, nome_deputado, partido, uf
  FROM gold_atlas_frentes_parlamentares
),
lista_votacoes AS (
  -- Obtendo os IDs de todas as votações que ocorreram na Bronze
  SELECT DISTINCT id_votacao 
  FROM bronze_presenca_votacoes
),
matriz_presenca_esperada AS (
  -- CROSS JOIN: Cria uma linha para cada deputado em cada votação
  SELECT * FROM total_deputados
  CROSS JOIN lista_votacoes
)

-- LEFT JOIN com a Bronze para ver quem de fato votou
SELECT 
    m.id_votacao,
    m.id_deputado,
    m.nome_deputado,
    m.partido,
    m.uf,
    CASE 
        WHEN b.voto IS NULL THEN 'AUSENTE' 
        ELSE 'PRESENTE' 
    END AS status_presenca,
    b.voto as tipo_voto  
FROM matriz_presenca_esperada m
LEFT JOIN bronze_presenca_votacoes b 
  ON m.id_deputado = b.id_deputado 
  AND m.id_votacao = b.id_votacao;

-- Ranking de faltas
SELECT nome_deputado, partido, COUNT(*) as total_faltas
FROM gold_monitor_absenteismo
WHERE status_presenca = 'AUSENTE'
GROUP BY ALL
ORDER BY total_faltas DESC;